# 4. Median of Two Sorted Arrays

**Difficulty:** Hard  
**Topics:** Array, Binary Search, Divide and Conquer  
**Language:** Python 3  
**Problem:** [LeetCode — Median of Two Sorted Arrays](https://leetcode.com/problems/median-of-two-sorted-arrays/)

Find the middle value of two sorted arrays considered together. Begin with a complete merge to make the median concrete, then locate only the partition around the middle using binary search. Each approach includes a short walkthrough, a correctness argument, complexity analysis, and a complete LeetCode-compatible implementation.

Run the notebook from top to bottom; only Python's standard library is needed. Edit the arrays in **Try your own input** to experiment. Copy the final code cell into the Python 3 editor for submission.

## 1. Problem

### Statement summary

Two integer arrays, `nums1` and `nums2`, are each sorted in nondecreasing order. Their lengths are `m` and `n`. Return the median of all their values considered in sorted order. Duplicates count as separate elements. At least one array is nonempty.

The solution must run in **O(log(m + n)) time**. The merge below is a learning baseline; the final binary partition algorithm meets the requirement.

### Constraints

- `m == len(nums1)` and `n == len(nums2)`
- `0 <= m <= 1000` and `0 <= n <= 1000`
- `1 <= m + n <= 2000`
- Every element of either array lies in `[-10**6, 10**6]`.
- Each input array is already sorted.

### Examples

**Example 1**

```text
Input: nums1 = [1,3], nums2 = [2]
Output: 2.00000
```

The combined ordering is `[1, 2, 3]`; its single middle value is `2`.

**Example 2**

```text
Input: nums1 = [1,2], nums2 = [3,4]
Output: 2.50000
```

The middle values in `[1, 2, 3, 4]` are `2` and `3`. Their average is `2.5`.

## 2. Understand the task

Return a numeric **median**, not an index and not the average of the two arrays' individual medians. The arrays may have different lengths, duplicates, or negative numbers. One array may be empty; both cannot be empty. Neither solution modifies its inputs.

For `N = m + n` values in sorted order:

- If `N` is odd, take the value at zero-based index `N // 2`.
- If `N` is even, average the values at indices `N // 2 - 1` and `N // 2`.

Use official example 2, `[1, 2]` and `[3, 4]`, for both walkthroughs. Merging reveals the middle values directly. The faster approach instead divides the values into a left part and a right part such that every left value is at most every right value. Then only the largest left value and, for an even total, the smallest right value matter.

The final algorithm assumes the supplied sortedness guarantee. Checking every element for sortedness inside that method would add linear work and violate the required logarithmic bound. Such validation belongs in the manual harness.

In [1]:
# Shared inputs for the two walkthroughs (official example 2).
EXAMPLE_NUMS1 = [1, 2]
EXAMPLE_NUMS2 = [3, 4]
EXAMPLE_EXPECTED = 2.5

## 3. Approach 1 — merge the sorted arrays

### Idea

Use two pointers to build the combined sorted sequence. At every step, take the smaller next available value. Once the merge is complete, read its one or two middle values.

### Algorithm

1. Start pointers `i = j = 0` and an empty `merged` list.
2. While either array has an unconsumed value, append the smaller next value. If an array is exhausted, take from the other one.
3. Advance only the pointer belonging to the appended value.
4. If the total is odd, return the middle value as a float.
5. Otherwise, return the average of the two middle values.

### Dry run

`nums1 = [1, 2]`, `nums2 = [3, 4]`

| Visit | Pointers before `(i, j)` | Value taken | `merged` afterward |
|---:|---|---:|---|
| 1 | (0, 0) | 1 from `nums1` | `[1]` |
| 2 | (1, 0) | 2 from `nums1` | `[1, 2]` |
| 3 | (2, 0) | 3 from `nums2` | `[1, 2, 3]` |
| 4 | (2, 1) | 4 from `nums2` | `[1, 2, 3, 4]` |

Both arrays are exhausted after four visits. `middle = 2`, so average `merged[1]` and `merged[2]`: `(2 + 3) / 2 = 2.5`.

### Why it works

Before each iteration, `merged` contains the smallest values already consumed, in sorted order. Since each input is sorted, the next smallest remaining value must be one of their current heads. Taking the smaller head, or the only available head, preserves sortedness and consumes exactly one value. The loop therefore produces the full combined ordering without losing duplicates. The odd/even middle-index formulas then return its median.

### Complexity

- **Worst-case time:** O(m + n). Each value is appended once; list append is amortized O(1).
- **Auxiliary space:** O(m + n), for the merged list.

This baseline does not satisfy the required logarithmic runtime. Stopping the merge just after the middle can save storage and some visits, but still requires linear time in the worst case.

### Implementation

In [2]:
class Solution:
    def findMedianSortedArrays(self, nums1: list[int], nums2: list[int]) -> float:
        m, n = len(nums1), len(nums2)
        if m + n == 0:
            raise ValueError("At least one array must be nonempty.")
        merged = []
        i = j = 0
        while i < m or j < n:
            if j == n or (i < m and nums1[i] <= nums2[j]):
                merged.append(nums1[i])
                i += 1
            else:
                merged.append(nums2[j])
                j += 1
        middle = (m + n) // 2
        if (m + n) % 2:
            return float(merged[middle])
        return (merged[middle - 1] + merged[middle]) / 2.0


MergeSolution = Solution
print(MergeSolution().findMedianSortedArrays(EXAMPLE_NUMS1, EXAMPLE_NUMS2))

2.5


In [3]:
# Teaching trace, separate from submission code.
def trace_merge(nums1, nums2):
    m, n = len(nums1), len(nums2)
    if m + n == 0:
        raise ValueError("At least one array must be nonempty.")
    merged = []
    i = j = 0
    while i < m or j < n:
        before = (i, j)
        if j == n or (i < m and nums1[i] <= nums2[j]):
            value, source = nums1[i], "nums1"
            merged.append(value)
            i += 1
        else:
            value, source = nums2[j], "nums2"
            merged.append(value)
            j += 1
        print(f"Visit {len(merged)}: pointers={before}, take {value} from {source}, merged={merged}")
    middle = (m + n) // 2
    result = (float(merged[middle]) if (m + n) % 2
              else (merged[middle - 1] + merged[middle]) / 2.0)
    print(f"Return {result}; values consumed = {m + n}")
    return result

assert trace_merge(EXAMPLE_NUMS1, EXAMPLE_NUMS2) == EXAMPLE_EXPECTED

Visit 1: pointers=(0, 0), take 1 from nums1, merged=[1]
Visit 2: pointers=(1, 0), take 2 from nums1, merged=[1, 2]
Visit 3: pointers=(2, 0), take 3 from nums2, merged=[1, 2, 3]
Visit 4: pointers=(2, 1), take 4 from nums2, merged=[1, 2, 3, 4]
Return 2.5; values consumed = 4


## 4. Approach 2 — binary search for a balanced partition

### From merging all values to locating the middle boundary

The median needs a boundary, not a fully merged array. Let `a` be the shorter array and `b` the longer. Swapping these local references does not copy or reorder either input.

Take `i` values from the start of `a` and `j` values from the start of `b` into the left part. Fix the left part's size as:

```text
left_size = (m + n + 1) // 2
i + j = left_size
j = left_size - i
```

For an odd total, the left part holds the extra value. Once `i` is chosen, `j` is determined, so there is just one search variable.

Read the values immediately beside the cuts:

```text
a: [ left part ... a_left | a_right ... right part ]
b: [ left part ... b_left | b_right ... right part ]
```

The arrays are internally sorted. Consequently, the whole left part is at most the whole right part exactly when both cross-array comparisons hold:

```text
a_left <= b_right
b_left <= a_right
```

At an empty left boundary use `-infinity`; at an empty right boundary use `+infinity`. These are comparison sentinels, not extra array elements. Conditional indexing is essential: Python's `a[-1]` is a real last element, not a missing boundary.

### Algorithm

1. Assign `a` to the shorter input; let `m = len(a)` and `n = len(b)`.
2. Search the inclusive range `0..m` for the cut `i`. These are element counts, so `m` itself is a legal cut.
3. Compute `j = left_size - i` and the four boundary values.
4. If both comparisons hold, return the largest left boundary for an odd total. For an even total, average the largest left and smallest right boundaries.
5. If `a_left > b_right`, too many values were taken from `a`; move `high` to `i - 1`.
6. Otherwise `b_left > a_right`; take more from `a` by moving `low` to `i + 1`.

Searching the shorter array ensures `0 <= j <= n` throughout `0 <= i <= m`: when `m <= n`, `m <= left_size <= n`. There is no need to allocate left/right slices.

### Dry run

`a = [1, 2]`, `b = [3, 4]`, `left_size = 2`

| Visit | Search range | `i`, `j` | `a_left`, `a_right` | `b_left`, `b_right` | Decision |
|---:|---|---|---|---|---|
| 1 | [0, 2] | 1, 1 | 1, 2 | 3, 4 | `3 > 2`: increase `i`, set `low = 2` |
| 2 | [2, 2] | 2, 0 | 2, +∞ | −∞, 3 | Both comparisons hold; return `(2 + 3) / 2 = 2.5` |

Stop immediately after the second partition check. No merged array is created.

### Why it works

**A valid partition exists.** Conceptually take the first `left_size` elements of a sorted merge. They consist of a prefix of each input (ties may be assigned consistently), so some legal `i, j` represent this split.

**The comparisons certify the split.** Within each array the left prefix is already at most the right suffix. The two cross-array comparisons cover the remaining possibilities, proving every value on the combined left is at most every value on the combined right. The side sizes are fixed by `i + j = left_size`.

**Each search step preserves a valid cut.** If `a_left > b_right`, increasing `i` would make `a_left` no smaller and, because `j` decreases, `b_right` no larger. Thus the current cut and every greater cut are invalid; a solution must be to the left. If `b_left > a_right`, decreasing `i` cannot fix the violation for the symmetric reason; a solution must be to the right. Removing the invalid half preserves at least one valid cut and strictly shrinks the search interval.

When a valid cut is found, the largest left value is the middle value for an odd total. For an even total, the largest left and smallest right values are exactly the two middle values. At least one input is nonempty, so the selected median boundaries are finite, even when one array is empty.

### Complexity

Let `p = min(len(nums1), len(nums2))` before swapping the local references.

- **Worst-case time:** O(log(p + 2)), because binary search considers `p + 1` possible cuts with constant work per visit. The `+2` notation includes the constant-time empty-array case. This satisfies the required logarithmic bound in the combined input size.
- **Auxiliary space:** O(1). Only references, indices, and boundary values are stored; there is no recursion and no slicing.

### Implementation

In [4]:
class Solution:
    def findMedianSortedArrays(self, nums1: list[int], nums2: list[int]) -> float:
        a, b = nums1, nums2
        if len(a) > len(b):
            a, b = b, a
        m, n = len(a), len(b)
        if m + n == 0:
            raise ValueError("At least one array must be nonempty.")

        left_size = (m + n + 1) // 2
        low, high = 0, m
        while low <= high:
            i = (low + high) // 2
            j = left_size - i
            a_left = a[i - 1] if i > 0 else float("-inf")
            a_right = a[i] if i < m else float("inf")
            b_left = b[j - 1] if j > 0 else float("-inf")
            b_right = b[j] if j < n else float("inf")

            if a_left <= b_right and b_left <= a_right:
                if (m + n) % 2:
                    return float(max(a_left, b_left))
                return (max(a_left, b_left) + min(a_right, b_right)) / 2.0
            if a_left > b_right:
                high = i - 1
            else:
                low = i + 1
        raise ValueError("No valid partition; inputs must be sorted.")


PartitionSolution = Solution
print(PartitionSolution().findMedianSortedArrays(EXAMPLE_NUMS1, EXAMPLE_NUMS2))

2.5


In [5]:
def trace_partition(nums1, nums2):
    a, b = nums1, nums2
    if len(a) > len(b):
        a, b = b, a
    m, n = len(a), len(b)
    if m + n == 0:
        raise ValueError("At least one array must be nonempty.")
    left_size = (m + n + 1) // 2
    low, high = 0, m
    visits = 0
    while low <= high:
        i = (low + high) // 2
        j = left_size - i
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < m else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < n else float("inf")
        visits += 1
        print(f"Visit {visits}: range=[{low}, {high}], i={i}, j={j}, "
              f"a=({a_left}, {a_right}), b=({b_left}, {b_right})")
        if a_left <= b_right and b_left <= a_right:
            result = (float(max(a_left, b_left)) if (m + n) % 2
                      else (max(a_left, b_left) + min(a_right, b_right)) / 2.0)
            print(f"  Balanced: return {result}; partition checks = {visits}")
            return result
        if a_left > b_right:
            high = i - 1
            print(f"  a_left > b_right: move left, high={high}")
        else:
            low = i + 1
            print(f"  b_left > a_right: move right, low={low}")
    raise ValueError("No valid partition; inputs must be sorted.")

assert trace_partition(EXAMPLE_NUMS1, EXAMPLE_NUMS2) == EXAMPLE_EXPECTED

Visit 1: range=[0, 2], i=1, j=1, a=(1, 2), b=(3, 4)
  b_left > a_right: move right, low=2
Visit 2: range=[2, 2], i=2, j=0, a=(2, inf), b=(-inf, 3)
  Balanced: return 2.5; partition checks = 2


## 5. Comparison and common mistakes

| Approach | Worst-case time | Auxiliary space | Main trade-off |
|---|---|---|---|
| Complete two-pointer merge | O(m + n) | O(m + n) | Easy to visualize, but misses the runtime requirement |
| Binary partition on the shorter array | O(log(min(m, n) + 2)) | O(1) | Meets the requirement; cut boundaries need care |

Choose binary partition for the final submission. Concatenating and sorting is another direct baseline, but ignores the sorted input structure and does not meet the logarithmic requirement either.

Common mistakes:

- Averaging the medians of the two arrays instead of selecting the combined middle rank.
- Requiring strictly smaller boundaries: use `<=` because duplicates are valid.
- Searching the longer array with the unrestricted `0..m` range, which can make `j` invalid.
- Treating cuts as element indices and omitting the legal cut at `m`.
- Reading index `-1` or `m` instead of using a sentinel at an empty side.
- Reversing the search direction after a failed comparison.
- Giving the extra element to the right while still returning `max(left)` for an odd total.
- Using integer division for the average of two middle values.

The methods reject two empty arrays as a convenience, but do not perform comprehensive input validation. In particular, the final fallback error is not a reliable detector of unsorted inputs: some invalid inputs can accidentally satisfy the boundary comparisons. The manual harness explicitly validates sortedness and all constraints before calling either method; that linear validation is separate from the partition algorithm's complexity.

## 6. Test the solutions

Test both implementations on all official examples and supplementary cases covering empty arrays in either position, odd and even totals, duplicate values, negative and extreme values, unequal lengths, and both search directions. Verify the return type and confirm neither input is modified. A maximum-size case exercises both arrays at length 1000.

In [6]:
from math import isclose

TEST_CASES = [
    ("Problem example 1", [1, 3], [2], 2.0),
    ("Problem example 2", [1, 2], [3, 4], 2.5),
    ("Supplementary: first empty", [], [1], 1.0),
    ("Supplementary: second empty", [2, 3], [], 2.5),
    ("Supplementary: singleton arrays", [1], [2], 1.5),
    ("Supplementary: all equal", [0, 0], [0, 0], 0.0),
    ("Supplementary: negatives", [-5, -3, -1], [-2], -2.5),
    ("Supplementary: move left", [5, 6], [1, 2, 3], 3.0),
    ("Supplementary: unequal lengths", [100], [1, 2, 3, 4, 5, 6], 4.0),
    ("Supplementary: value bounds", [-10**6], [10**6], 0.0),
    ("Supplementary: empty and duplicates", [], [2, 2, 2, 3], 2.0),
    ("Supplementary: maximum lengths", list(range(1000)), list(range(1000, 2000)), 999.5),
]
for name, nums1, nums2, expected in TEST_CASES:
    before1, before2 = nums1.copy(), nums2.copy()
    for implementation in (MergeSolution, PartitionSolution):
        result = implementation().findMedianSortedArrays(nums1, nums2)
        assert type(result) is float and isclose(result, expected, rel_tol=0, abs_tol=1e-9), (name, result)
        assert nums1 == before1 and nums2 == before2
    print(f"PASS | {name}: {expected}")

PASS | Problem example 1: 2.0
PASS | Problem example 2: 2.5
PASS | Supplementary: first empty: 1.0
PASS | Supplementary: second empty: 2.5
PASS | Supplementary: singleton arrays: 1.5
PASS | Supplementary: all equal: 0.0
PASS | Supplementary: negatives: -2.5
PASS | Supplementary: move left: 3.0
PASS | Supplementary: unequal lengths: 4.0
PASS | Supplementary: value bounds: 0.0
PASS | Supplementary: empty and duplicates: 2.0
PASS | Supplementary: maximum lengths: 999.5


## 7. Try your own input

Edit `nums1_text`, `nums2_text`, and optionally `approach` in the next cell, then run it again. Use JSON arrays such as `[1, 3]`; select `"partition"` or `"merge"`. Empty arrays use `[]`, but at least one input must contain a value.

The harness checks list types, lengths, integer bounds, and nondecreasing order. It rejects booleans and decimal values rather than treating them as integers. It uses safe JSON parsing and produces explanatory errors without blocking **Run All**. Validation scans both arrays in O(m + n) time and JSON parsing allocates the inputs; neither is part of the final submission's constant-space logarithmic algorithm.

In [7]:
import json

def validate_arrays(nums1, nums2):
    for name, values in (("nums1", nums1), ("nums2", nums2)):
        if not isinstance(values, list) or len(values) > 1000:
            raise ValueError(f"{name} must be a JSON array with 0 to 1000 integers.")
        if any(type(value) is not int or not -10**6 <= value <= 10**6 for value in values):
            raise ValueError(f"{name} values must be integers from -10^6 to 10^6.")
        if any(values[i] > values[i + 1] for i in range(len(values) - 1)):
            raise ValueError(f"{name} must be sorted in nondecreasing order.")
    if not nums1 and not nums2:
        raise ValueError("At least one array must be nonempty.")


def run_manual(nums1_text, nums2_text, approach="partition"):
    try:
        nums1, nums2 = json.loads(nums1_text), json.loads(nums2_text)
        validate_arrays(nums1, nums2)
        implementations = {"merge": MergeSolution, "partition": PartitionSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'merge' or 'partition'.")
        result = implementations[approach]().findMedianSortedArrays(nums1, nums2)
        print(f"{approach}: lengths=({len(nums1)}, {len(nums2)}); median={result}")
        return result
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
nums1_text = "[1, 2]"
nums2_text = "[3, 4]"
approach = "partition"  # Or "merge".
manual_result = run_manual(nums1_text, nums2_text, approach)

partition: lengths=(2, 2); median=2.5


In [8]:
assert run_manual("[1, 3]", "[2]", "merge") == 2.0
assert run_manual("[]", "[-2, -1]") == -1.5
assert run_manual("[]", "[]") is None
assert run_manual("[2, 1]", "[3]") is None
assert run_manual("[true]", "[2]") is None
assert run_manual("[1.0]", "[2]") is None
assert run_manual("null", "[2]") is None
assert run_manual("not JSON", "[2]") is None
assert run_manual("[1000001]", "[]") is None
assert run_manual(json.dumps([0] * 1001), "[1]") is None
assert run_manual("[1]", "[2]", "unknown") is None
print("Manual-input harness checks passed.")

merge: lengths=(2, 1); median=2.0
partition: lengths=(0, 2); median=-1.5
Input error: At least one array must be nonempty.
Input error: nums1 must be sorted in nondecreasing order.
Input error: nums1 values must be integers from -10^6 to 10^6.
Input error: nums1 values must be integers from -10^6 to 10^6.
Input error: nums1 must be a JSON array with 0 to 1000 integers.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: nums1 values must be integers from -10^6 to 10^6.
Input error: nums1 must be a JSON array with 0 to 1000 integers.
Input error: approach must be 'merge' or 'partition'.
Manual-input harness checks passed.


## 8. Final submission

Copy the next cell into LeetCode's Python 3 editor. It uses the official `Solution.findMedianSortedArrays(self, nums1: list[int], nums2: list[int]) -> float` interface, performs binary search on the shorter input, and requires no notebook helpers or imports.

In [9]:
class Solution:
    def findMedianSortedArrays(self, nums1: list[int], nums2: list[int]) -> float:
        a, b = nums1, nums2
        if len(a) > len(b):
            a, b = b, a
        m, n = len(a), len(b)
        if m + n == 0:
            raise ValueError("At least one array must be nonempty.")

        left_size = (m + n + 1) // 2
        low, high = 0, m
        while low <= high:
            i = (low + high) // 2
            j = left_size - i
            a_left = a[i - 1] if i > 0 else float("-inf")
            a_right = a[i] if i < m else float("inf")
            b_left = b[j - 1] if j > 0 else float("-inf")
            b_right = b[j] if j < n else float("inf")

            if a_left <= b_right and b_left <= a_right:
                if (m + n) % 2:
                    return float(max(a_left, b_left))
                return (max(a_left, b_left) + min(a_right, b_right)) / 2.0
            if a_left > b_right:
                high = i - 1
            else:
                low = i + 1
        raise ValueError("No valid partition; inputs must be sorted.")